In [1]:
print(123)

123


In [2]:
# MODULE 1

query = "I just discovered the course. Can I still join?"

"""
What is RAG again? 


 user has Question Q -> ASSISTANT <--> FAQ (DOCUMENTS)
 The assistant is sending a request to a database where the faq data is and then its getting back some docyuemnts that potentiall
 contain the answer to our question, which dont know in advance waht documents will be useful for the question, but we assume that 
 top 5 search results for the search will be useful. 

 assistant sends PROMPT (Q:Q, CONTEXT: d1 ... d5) -> LLM 
"""




'\nWhat is RAG again? \n\n\n user has Question Q -> ASSISTANT <--> FAQ (DOCUMENTS)\n The assistant is sending a request to a database where the faq data is and then its getting back some docyuemnts that potentiall\n contain the answer to our question, which dont know in advance waht documents will be useful for the question, but we assume that \n top 5 search results for the search will be useful. \n\n assistant sends PROMPT (Q:Q, CONTEXT: d1 ... d5) -> LLM \n'

In [3]:
# TEXT SEARCH 

query = "I just discovered the course. Can I still join?"

"""
# when we decompose it into meaningful things 
# discover, course, join?

then we look for docs that contain ideally all 3, or at least 2 or 1 of these words 
and if it doesnt contain any of these words we dont rturn anything. 

"""



query = "I just found out about the progrma, can I still enroll?"
"""
Semantically the questions are the same but the words are completely different.
find, program, enroll.

text search would fail and tats why we are doing vector search.
"""


'\nSemantically the questions are the same but the words are completely different.\nfind, program, enroll.\n\ntext search would fail and tats why we are doing vector search.\n'

In [4]:
# word2vec

"""


enroll              w = [0.5, 0.7, -0.2, ...]  (lenth can be large e.g. 512)
join


we take a word and turn this into bunch of numbers. 
this document can be very large.
turning this word into a vector is called embedding. 

the idea is that enroll and join should have similar numbers, and semantically close in the vector space and thats why they are close to each other in the vector space (for simplicity assume 3 dim for example.)
the word docker should be far from those. 

we call this word embedding.
we can do the same with entire sentences! 
sentences with similar meaning e.g. Q1, Q2 

"can i still join course / can i still enroll into the program" are semantically close and 
"how to run docker" is semantically  far apart.

we will have 500 different documents from the faq vectorspace db

then we will return the 5 documents that are closest to the user question in vectorspace.


ALSO CALLED SEMANTIC SEARCH WHERE WE LOOK INTO THE MEANING AS OPPOSED TO TEXT SEARCH.

STEP 1: TURNING QUERIES INTO VECTORS
STEP 2: PERFORM SEARCH

In text search is only 1 step
"""


'\n\n\nenroll              w = [0.5, 0.7, -0.2, ...]  (lenth can be large e.g. 512)\njoin\n\n\nwe take a word and turn this into bunch of numbers. \nthis document can be very large.\nturning this word into a vector is called embedding. \n\nthe idea is that enroll and join should have similar numbers, and semantically close in the vector space and thats why they are close to each other in the vector space (for simplicity assume 3 dim for example.)\nthe word docker should be far from those. \n\nwe call this word embedding.\nwe can do the same with entire sentences! \nsentences with similar meaning e.g. Q1, Q2 \n\n"can i still join course / can i still enroll into the program" are semantically close and \n"how to run docker" is semantically  far apart.\n\nwe will have 500 different documents from the faq vectorspace db\n\nthen we will return the 5 documents that are closest to the user question in vectorspace.\n\n\nALSO CALLED SEMANTIC SEARCH WHERE WE LOOK INTO THE MEANING AS OPPOSED TO T

In [5]:
# SBERT.net = sentenceTransformers documentation this is doing the embedding 

# there are many models we chose a small one among pretrained models it has 80mb so itll be faster
# the larger the model the slower it is.

# the model itself is stored in huggingface  
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# simple example 

q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [7]:
v1.shape

# 384 numbers in that 384 dimensional embedding, each number has some meaning representing the concept that is internal to the neural network that is embedding our models into this vectorspace

(384,)

In [8]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [9]:
# now we will take both query and document vectors and look at the similarity: 

# for that we compute dot product (vector vector multiplication;)

# w^T d those 2 vectors are normalized (unit vectors for which the length is always exactly 1) 
# multiplying these 2 unit vectors we get cosine similarity


In [10]:
v1.dot(dv)

np.float32(0.32332397)

In [11]:
# other example

q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

v2.dot(dv)

# 0 means they are super different (orthogonal = they dont have anything in common)



np.float32(0.019730438)

In [12]:
# SO THE IDEA IS TO TAKE THE similarity between the q and all the x in the vector space and take the top 5!

# 3. EMBEDDING OUR DATASET

In [13]:

# dowlodin helper script inget 

!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py 

--2026-06-29 23:19:05--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-29 23:19:05 (42.8 MB/s) - ‘ingest.py.1’ saved [888/888]



In [14]:
from ingest import load_faq_data
documents = load_faq_data()

In [15]:
documents[10]

# we need to turn this python dictionary and turn it into proper text before being able to embed this

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [16]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [17]:
texts [10]

# thee text has 1200 texts or something

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [18]:
len(texts)

# we want to split this into batches instead of throwing everything at the same time into the vector space

1350

In [19]:
# embedding into the vectorspace 

from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [20]:
# lopp we will implement something similar like this but fastr

scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [21]:
scores[:10]

[np.float32(0.48740578),
 np.float32(0.20991932),
 np.float32(0.76294106),
 np.float32(0.4437854),
 np.float32(0.2608399),
 np.float32(0.48665166),
 np.float32(0.30061564),
 np.float32(0.56009996),
 np.float32(0.4596049),
 np.float32(0.2562817)]

In [22]:
# we turne the vectors into a matrix X (rows are documents)

import numpy as np
X = np.array(vectors)


In [23]:
X.shape

(1350, 384)

In [24]:
# 4. VECTOR SEARCH 

# we will use matrix vector multiplication which is very fast it is optimized

#

scores = X.dot(v1)



In [25]:
scores 

array([ 0.48740575,  0.20991933,  0.762941  , ..., -0.08637968,
        0.03759793, -0.03037044], shape=(1350,), dtype=float32)

In [26]:
# numpy used to select the closest ones (argmax select the highest value elment index)

idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [27]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [28]:
# WE SELECT THE TOP 5 WITH NUMPY

# we want to have more than top 1 because maybe the answer is scattered across more than 1 answers
# the number 5 choice is a gut feeling

# instead of using argmax: 

top5 = np.argsort(scores)[-5:]  # gives sorted values; then give top 5 largest 
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [29]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [30]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192132
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [31]:
# -> apparently we retrieved the same question from different courses 

In [32]:
# TRICK

 # instead of np.argsort(-scores)[-5:]: 
np.argsort(-scores)[:5]

array([  2, 625, 907, 538,   7])

In [33]:
# 5. vector search with minsearch alexeys library

"""
The judge ruled out the possibility of a crime

lets use llm as a judge approach to evaluate our llm

-> here judge has different contexts
-> thanks to embedding the model can understand judge is related to some other words in the sentence
"""


'\nThe judge ruled out the possibility of a crime\n\nlets use llm as a judge approach to evaluate our llm\n\n-> here judge has different contexts\n-> thanks to embedding the model can understand judge is related to some other words in the sentence\n'

In [34]:
# we just did all of these low level things manually but now will use a library

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])  # keyword_fields arg has the purpose of being able to do keywrod search with that later
vindex.fit(X, documents)

In [35]:
vindex.search(v1)  # num_results=5, filter_dict=['course': 'llm-zoomcamp']

[{'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'doc_id': '3f1424af17'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.",
  'doc_id': '2d8b16c2a0'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'quest

In [36]:
results = vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})  # 
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [ ]:
# 6 - RAG with Vector Search
